In [6]:
import pandas as pd

# 1. Load the dataset.
df = pd.read_csv('data/ecommerce_orders_raw.csv')
# 2. Display its structure and dimensions.
print('Shape (rows, columns):', df.shape)
df.info()

# 3. Identify the data types of all columns.
print(df.dtypes)

# 4. Identify missing values.
missing = df.isnull().sum()
print(missing[missing > 0])

# 5. Identify duplicate orders.
print('Total rows:', len(df))

full_dupes = df.duplicated().sum()
print('Fully duplicate rows:', full_dupes)

order_id_dupes = df.duplicated(subset=['Order_ID']).sum()
print('Duplicate Order_IDs:', order_id_dupes)

# 6. Examine the unique values in:
# o City
# o Product Category
# o Payment Method 
for col in ['City', 'Product_Category', 'Payment_Method']:
    print(f'=== {col} ===')
    print(df[col].value_counts())
    print()

Shape (rows, columns): (15100, 11)
<class 'pandas.DataFrame'>
RangeIndex: 15100 entries, 0 to 15099
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          15100 non-null  str    
 1   Customer_ID       15100 non-null  str    
 2   City              15100 non-null  str    
 3   Product_Category  15100 non-null  str    
 4   Product           15100 non-null  str    
 5   Quantity          15100 non-null  int64  
 6   Unit_Price        15100 non-null  float64
 7   Discount          15100 non-null  int64  
 8   Payment_Method    15100 non-null  str    
 9   Order_Date        15100 non-null  str    
 10  Rating            14699 non-null  float64
dtypes: float64(2), int64(2), str(7)
memory usage: 1.3 MB
Order_ID                str
Customer_ID             str
City                    str
Product_Category        str
Product                 str
Quantity              int64
Unit_Price          float64
Di

In [11]:
# 7. Handle missing ratings.
median_rating = df['Rating'].median()
df['Rating'] = df['Rating'].fillna(median_rating)

print('Missing after:', df['Rating'].isnull().sum())

# 8. Standardize city names.
df['City'] = df['City'].str.strip().str.title()

print(df['City'].value_counts())

# 9. Identify invalid quantities.
print(df['Quantity'].describe())
print()

invalid_qty = df[df['Quantity'] <= 0]
print('Invalid quantity records:', len(invalid_qty))
print(invalid_qty['Quantity'].value_counts())

# 10. Handle duplicate orders.
print('Rows before:', len(df))
df = df.drop_duplicates()
print('Rows after:', len(df))

# 11. Convert Order_Date into an appropriate datetime format. 
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

print(df['Order_Date'].dtype)
print(df['Order_Date'].min(), 'to', df['Order_Date'].max())

Missing after: 0
City
Chennai       2164
Pune          1863
Hyderabad     1852
Mumbai        1841
Kolkata       1835
Bangalore     1831
Coimbatore    1826
Delhi         1788
Name: count, dtype: int64
count    15000.000000
mean         3.016067
std          1.429899
min         -2.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          5.000000
Name: Quantity, dtype: float64

Invalid quantity records: 20
Quantity
-2    20
Name: count, dtype: int64
Rows before: 15000
Rows after: 15000
datetime64[ns]
2025-01-01 00:00:00 to 2026-07-31 00:00:00


In [14]:
# 12. Create:
# Gross_Amount = Quantity × Unit_Price
df['Gross_Amount'] = df['Quantity'] * df['Unit_Price']

# 13. Create:
# Discount_Amount = Gross_Amount × Discount / 100
df['Discount_Amount'] = df['Gross_Amount'] * df['Discount'] / 100

# 14. Create:
# Net_Amount = Gross_Amount - Discount_Amount
df['Net_Amount'] = df['Gross_Amount'] - df['Discount_Amount']

print(df[['Gross_Amount', 'Discount_Amount', 'Net_Amount']].head())

# 15. Create a Rating_Category column:
# 1–2 → Poor
# 3 → Average
# 4 → Good
# 5 → Excellent
def rating_category(r):
    if r <= 2:
        return 'Poor'
    elif r == 3:
        return 'Average'
    elif r == 4:
        return 'Good'
    else:
        return 'Excellent'

df['Rating_Category'] = df['Rating'].apply(rating_category)
print(df['Rating_Category'].value_counts())

# 16. Extract the following from Order_Date:
# • Year
# • Month
# • Day
# • Day of Week
df['Year'] = df['Order_Date'].dt.year
df['Month'] = df['Order_Date'].dt.month
df['Day'] = df['Order_Date'].dt.day
df['Day_of_Week'] = df['Order_Date'].dt.day_name()

print(df[['Order_Date','Year','Month','Day','Day_of_Week']].head())

   Gross_Amount  Discount_Amount   Net_Amount
0      20084.16        2008.4160   18075.7440
1      34791.00        5218.6500   29572.3500
2      31386.00        4707.9000   26678.1000
3     363504.30       72700.8600  290803.4400
4     448564.45      112141.1125  336423.3375
Rating_Category
Good         5439
Excellent    5144
Average      2511
Poor         1906
Name: count, dtype: int64
                     Order_Date  Year  Month  Day Day_of_Week
0 2026-07-25 02:03:50.895393024  2026      7   25    Saturday
1 2026-06-27 05:51:34.526301752  2026      6   27    Saturday
2 2026-07-25 15:53:20.613374224  2026      7   25    Saturday
3 2025-02-03 22:44:55.379691979  2025      2    3      Monday
4 2025-01-16 18:48:10.272684845  2025      1   16    Thursday


In [22]:
# 17. Calculate total revenue.
df['Gross_Amount'] = df['Quantity'] * df['Unit_Price']
df['Discount_Amount'] = df['Gross_Amount'] * df['Discount'] / 100
df['Net_Amount'] = df['Gross_Amount'] - df['Discount_Amount']

total_revenue = df['Net_Amount'].sum()
print(f'Total Revenue: ₹{total_revenue:,.2f}')

# 18. Calculate total number of orders.
total_orders = df['Order_ID'].nunique()
print('Total number of orders:', total_orders)

# 19. Find the average order value.
df['Gross_Amount'] = df['Quantity'] * df['Unit_Price']
df['Discount_Amount'] = df['Gross_Amount'] * df['Discount'] / 100
df['Net_Amount'] = df['Gross_Amount'] - df['Discount_Amount']

avg_order_value = df['Net_Amount'].mean()
print(f'Average Order Value: ₹{avg_order_value:,.2f}')

# 20. Find total revenue by product category.
df['Gross_Amount'] = df['Quantity'] * df['Unit_Price']
df['Discount_Amount'] = df['Gross_Amount'] * df['Discount'] / 100
df['Net_Amount'] = df['Gross_Amount'] - df['Discount_Amount']

revenue_by_category = df.groupby('Product_Category')['Net_Amount'].sum().sort_values(ascending=False)
print(revenue_by_category.round(2))

# 21. Find the top 10 products by revenue.
df['Gross_Amount'] = df['Quantity'] * df['Unit_Price']
df['Discount_Amount'] = df['Gross_Amount'] * df['Discount'] / 100
df['Net_Amount'] = df['Gross_Amount'] - df['Discount_Amount']

top10_products = df.groupby('Product')['Net_Amount'].sum().sort_values(ascending=False).head(10)
print(top10_products)

# 22. Find the city generating the highest revenue.
df['Gross_Amount'] = df['Quantity'] * df['Unit_Price']
df['Discount_Amount'] = df['Gross_Amount'] * df['Discount'] / 100
df['Net_Amount'] = df['Gross_Amount'] - df['Discount_Amount']

revenue_by_city = df.groupby('City')['Net_Amount'].sum().sort_values(ascending=False)
print('Highest revenue city:', revenue_by_city.idxmax())

# 23. Find the most frequently used payment method.
payment_counts = df['Payment_Method'].value_counts()
print('Most frequent:', payment_counts.idxmax(), '-', payment_counts.max(), 'orders')

# 24. Find the average rating for each product category.
avg_rating_by_category = df.groupby('Product_Category')['Rating'].mean().sort_values(ascending=False)
print(avg_rating_by_category.round(2))

# 25. Calculate monthly revenue.
df['Year_Month'] = df['Order_Date'].dt.to_period('M')
monthly_revenue = df.groupby('Year_Month')['Net_Amount'].sum()
print(monthly_revenue)

# 26. Identify the highest-revenue month.
df['Year_Month'] = df['Order_Date'].dt.to_period('M')
monthly_revenue = df.groupby('Year_Month')['Net_Amount'].sum()

print('Highest revenue month:', monthly_revenue.idxmax())
print('Revenue:', monthly_revenue.max())
# 27. Find the top 10 customers by total spending.
top10_customers = df.groupby('Customer_ID')['Net_Amount'].sum().sort_values(ascending=False).head(10)
print(top10_customers)

# 28. Compare revenue generated by different cities.
city_comparison = df.groupby('City').agg(
    Total_Revenue=('Net_Amount', 'sum'),
    Total_Orders=('Order_ID', 'count'),
    Avg_Order_Value=('Net_Amount', 'mean')
).sort_values('Total_Revenue', ascending=False)

print(city_comparison)

# 29. Find which product category has the highest average order value.
aov_by_category = df.groupby('Product_Category')['Net_Amount'].mean().sort_values(ascending=False)
print('Highest AOV category:', aov_by_category.idxmax())

Total Revenue: ₹1,979,711,486.41
Total number of orders: 15000
Average Order Value: ₹131,980.77
Product_Category
Clothing           3.451293e+08
Books              3.325953e+08
Beauty             3.307883e+08
Sports             3.295151e+08
Electronics        3.231161e+08
Home Appliances    3.185674e+08
Name: Net_Amount, dtype: float64
Product
Jeans         9.337433e+07
Jacket        8.881844e+07
Yoga Mat      8.832334e+07
Self Help     8.825103e+07
Shampoo       8.816100e+07
Skin Serum    8.718235e+07
Laptop        8.697328e+07
Mixer         8.515870e+07
Headphones    8.354481e+07
Textbook      8.347781e+07
Name: Net_Amount, dtype: float64
Highest revenue city: Chennai
Most frequent: Debit Card - 3062 orders
Product_Category
Clothing           3.91
Beauty             3.90
Sports             3.88
Home Appliances    3.85
Books              3.85
Electronics        3.83
Name: Rating, dtype: float64
Year_Month
2025-01    1.077725e+08
2025-02    9.685231e+07
2025-03    1.057524e+08
2025-04 